In [ ]:
# how to import and use the metrics
from sklearn.metrics import accuracy_score
accuracy_score(y_true, y_pred)

from sklearn.metrics import mean_squared_error
mean_squared_error(y_true, y_pred)


In [ ]:
# basic checks on raw data
df.shape
df.info()
df.describe()
df.isna().mean()

# drop rows with missing target
df = df.dropna(subset=["target"]).copy()

# exact duplicates
df = df.drop_duplicates()

# replace obvious placeholders with nan (check -1 vs real values first)
df = df.replace(-999, np.nan)

In [ ]:
# parse timestamp and sort chronologically before splitting or building folds
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

# check date range, duplicates, and time gaps
print(df["timestamp"].min(), "->", df["timestamp"].max())
print("duplicate timestamps:", df["timestamp"].duplicated().sum())
print(df["timestamp"].diff().value_counts().head())

# basic calendar features (no target or future info)
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = df["dayofweek"] >= 5

# past-only lag/rolling features: always shift(1) so current row isn't leaked
df["y_lag_1"] = df["target"].shift(1)
df["y_roll_mean_24"] = df["target"].shift(1).rolling(24).mean()

In [ ]:
# average numeric cols per id, keep first for non-numeric
num_cols = df.select_dtypes(include=np.number).columns.difference(["id", "target"])
cat_cols = df.select_dtypes(exclude=np.number).columns.difference(["id"])

agg_map = {c: "mean" for c in num_cols}
agg_map.update({c: "first" for c in cat_cols})
agg_map["target"] = "mean"

df_clean = df.groupby("id", as_index=False).agg(agg_map)

In [ ]:
# check correlations on train set
X_train.corr(numeric_only=True)

# May also use MI

# 2d quantile table to spot interactions
tmp = X_train[["f1", "f2"]].assign(target=y_train).copy()
tmp["f1_q"] = pd.qcut(tmp["f1"], q=5, duplicates="drop")
tmp["f2_q"] = pd.qcut(tmp["f2"], q=5, duplicates="drop")

mean_grid = tmp.pivot_table(index="f2_q", columns="f1_q", values="target", aggfunc="mean")
count_grid = tmp.pivot_table(index="f2_q", columns="f1_q", values="target", aggfunc="count")

In [ ]:
# ColumnTransformer + Pipeline syntax
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")), # drop="first" only for unregularized OLS
])

preprocessor = ColumnTransformer(
    transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
    remainder="drop",
)

In [ ]:
# General CV & GridSearchCV syntax
cv = KFold(n_splits=5, shuffle=True, random_state=42) # StratifiedKFold for classification
# scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")

pipe = Pipeline([("prep", preprocessor), ("model", Ridge())])
gs = GridSearchCV(
    pipe,
    param_grid={"model__alpha": [0.1, 1.0, 10.0]},
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
gs.fit(X_train, y_train)
y_pred = gs.predict(X_test)

In [ ]:
# StackingRegressor syntax (requires partitioning cv like KFold)
stack = StackingRegressor(
    estimators=[("ridge", Ridge()), ("rf", RandomForestRegressor(random_state=42))],
    final_estimator=Ridge(alpha=1.0, positive=True),
    cv=cv,
)
stack.fit(X_train, y_train)
y_pred = stack.predict(X_test)

In [ ]:
# Residual & Q-Q plot syntax
resid = y_true - y_pred
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(y_pred, resid, alpha=0.5)
axes[0].axhline(0, linestyle="--")
stats.probplot(resid, dist="norm", plot=axes[1])
plt.tight_layout()

In [ ]:
# Regression metrics
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

In [ ]:
# Classification metrics
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, roc_auc_score, average_precision_score,
)

print(confusion_matrix(y_true, y_pred))          # TN FP / FN TP
print(classification_report(y_true, y_pred))     # precision/recall/F1 per class

# F1: use macro for class-balanced, weighted for imbalanced
f1_macro    = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

# ROC-AUC needs probability scores, not hard labels
y_prob = model.predict_proba(X_test)[:, 1]  # binary
roc_auc = roc_auc_score(y_true, y_prob)
roc_auc_score(y_true, y_prob_matrix, multi_class="ovr", average="macro")  # multiclass

# PR-AUC: better than ROC-AUC under heavy class imbalance
pr_auc = average_precision_score(y_true, y_prob)